# 06. Final Comparison of All Approaches

**Project:** Road Sign Recognition Using Deep Learning (Cambodian Traffic Signs, CamTSD), 26 classes

This notebook **does not train anything**. It reads the results already saved by the experiment notebooks and compares them side by side:

| Notebook | Approach | Saved results read here |
|---|---|---|
| `02_cnn_from_scratch` | 1. SmallCNN trained from scratch | `results/runs/small_cnn/<run>/metrics/results.json` |
| `03_resnet18_frozen` | 2. ResNet-18, frozen pretrained backbone | `results/runs/resnet18_frozen/<run>/metrics/results.json` |
| `04_resnet18_finetuned` | 3. ResNet-18, fully fine-tuned | `results/runs/resnet18_finetuned/<run>/metrics/results.json` |
| `05_hyperparameter_tuning` | 3. tuned: **final selected model** | `results/runs/hyperparameter_tuning/<run>/metrics/final_metrics.json` |

Every approach used the **same leak-free split** (2,153 train / 454 validation / 463 test crops, 26 classes, no photo shared between splits), the same preprocessing and seed 42. Every model was selected on validation data, and the test set was used once per model, after selection. That is what makes the numbers below directly comparable.

## 1. Setup

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- Edit these if your setup differs ------------------------------------
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")

# Leave as None to use the most recent run of each experiment, or pin a run folder name,
# e.g. {"small_cnn": "20261006-143228_seed42"}.
RUN_OVERRIDES = {"small_cnn": None, "resnet18_frozen": None, "resnet18_finetuned": None,
                 "hyperparameter_tuning": None}
# --------------------------------------------------------------------------

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_ROOT = DRIVE_PROJECT_DIR / "results" / "runs"
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RESULTS_ROOT = PROJECT_ROOT / "results" / "runs"

# Optional override (used for local verification).
RESULTS_ROOT = Path(os.environ.get("FINAL_COMPARISON_RESULTS_ROOT", RESULTS_ROOT))

# Outputs of this notebook go to a NEW folder; existing experiment results are only read.
COMPARISON_DIR = RESULTS_ROOT / "final_comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading results from: {RESULTS_ROOT}")
print(f"Writing comparison to: {COMPARISON_DIR}")

## 2. Load the saved results

For each experiment the most recent run folder is used, unless `RUN_OVERRIDES` pins a specific one. The cell then checks that all four results are comparable: same test-set size, 26 classes, seed 42, and the expected experiment names.

In [ ]:
def pick_run(experiment, results_file):
    # Most recent run folder of an experiment that contains the given results file.
    exp_dir = RESULTS_ROOT / experiment
    pinned = RUN_OVERRIDES.get(experiment)
    if pinned:
        path = exp_dir / pinned / "metrics" / results_file
        if not path.is_file():
            raise FileNotFoundError(f"Pinned run not found: {path}")
        return path
    candidates = sorted(exp_dir.glob(f"*/metrics/{results_file}"))
    if not candidates:
        raise FileNotFoundError(f"No {results_file} found under {exp_dir}. Has this experiment been run?")
    return candidates[-1]


def load_json(path):
    with open(path) as f:
        return json.load(f)


paths_used = {
    "small_cnn": pick_run("small_cnn", "results.json"),
    "resnet18_frozen": pick_run("resnet18_frozen", "results.json"),
    "resnet18_finetuned": pick_run("resnet18_finetuned", "results.json"),
    "hyperparameter_tuning": pick_run("hyperparameter_tuning", "final_metrics.json"),
}
results = {name: load_json(path) for name, path in paths_used.items()}
for name, path in paths_used.items():
    print(f"{name:22s} <- {path.relative_to(RESULTS_ROOT).as_posix()}")

# ---- consistency checks: all approaches must be comparable ----
EXPECTED_EXPERIMENTS = {
    "small_cnn": "approach1_small_cnn_from_scratch",
    "resnet18_frozen": "approach2_resnet18_frozen_backbone",
    "resnet18_finetuned": "approach3_resnet18_fully_finetuned",
    "hyperparameter_tuning": "approach3_resnet18_fully_finetuned_hyperparameter_tuning",
}
for name, expected in EXPECTED_EXPERIMENTS.items():
    assert results[name]["experiment"] == expected, f"{name}: unexpected experiment {results[name]['experiment']}"

tuned = results["hyperparameter_tuning"]
test_sizes = {name: results[name]["test"]["num_test_samples"] for name in ("small_cnn", "resnet18_frozen", "resnet18_finetuned")}
test_sizes["hyperparameter_tuning"] = tuned["final_test_metrics"]["num_test_samples"]
assert set(test_sizes.values()) == {463}, f"test-set sizes differ: {test_sizes}"
for name in ("small_cnn", "resnet18_frozen", "resnet18_finetuned"):
    ds = results[name]["dataset"]
    assert (ds["train_samples"], ds["val_samples"], ds["test_samples"], ds["num_classes"]) == (2153, 454, 463, 26), name
    assert results[name]["seed"] == 42, name
assert tuned["seed"] == 42

print("\nConsistency checks passed: all four models were evaluated on the same 463-image test set "
      "(26 classes, seed 42, split 2153 / 454 / 463).")

## 3. Final results table

In [ ]:
def baseline_row(approach, res):
    return {
        "Approach": approach,
        "Test Accuracy": res["test"]["accuracy"],
        "Macro Precision": res["test"]["precision_macro"],
        "Macro Recall": res["test"]["recall_macro"],
        "Macro F1": res["test"]["f1_macro"],
        "Weighted F1": res["test"]["f1_weighted"],
        "Trainable Parameters": res["model"]["trainable_parameters"],
        "Training Time (min)": res["training"]["total_training_time_sec"] / 60,
        "Best Val Accuracy": res["training"]["best_val_acc"],
    }


best_cfg = tuned["best_config"]
TUNED_LABEL = (f"3. ResNet-18 fine-tuned, tuned (config {best_cfg['config']}: "
               f"lr={best_cfg['learning_rate']:g}, wd={best_cfg['weight_decay']:g}) - FINAL MODEL")

rows = [
    baseline_row("1. SmallCNN from scratch", results["small_cnn"]),
    baseline_row("2. ResNet-18 frozen backbone", results["resnet18_frozen"]),
    baseline_row("3. ResNet-18 fully fine-tuned (baseline)", results["resnet18_finetuned"]),
    {
        "Approach": TUNED_LABEL,
        "Test Accuracy": tuned["final_test_metrics"]["accuracy"],
        "Macro Precision": tuned["final_test_metrics"]["precision_macro"],
        "Macro Recall": tuned["final_test_metrics"]["recall_macro"],
        "Macro F1": tuned["final_test_metrics"]["f1_macro"],
        "Weighted F1": tuned["final_test_metrics"]["f1_weighted"],
        "Trainable Parameters": tuned["model"]["trainable_parameters"],
        "Training Time (min)": best_cfg["training_time_sec"] / 60,
        "Best Val Accuracy": best_cfg["best_val_acc"],
    },
]
results_df = pd.DataFrame(rows)
results_df.to_csv(COMPARISON_DIR / "final_results_table.csv", index=False)

display_df = results_df.copy()
for col in ("Test Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1", "Best Val Accuracy"):
    display_df[col] = display_df[col].map("{:.2%}".format)
display_df["Trainable Parameters"] = display_df["Trainable Parameters"].map("{:,}".format)
display_df["Training Time (min)"] = display_df["Training Time (min)"].map("{:.1f}".format)
print(display_df.to_string(index=False))

total_tuning_min = sum(r["training_time_sec"] for r in tuned["tuning_results"]) / 60
print(f"\nTraining time for the tuned model = its selected run (config {best_cfg['config']}). "
      f"The whole 4-configuration search took {total_tuning_min:.1f} min.")
print(f"Saved {COMPARISON_DIR / 'final_results_table.csv'}")

## 4. Comparison figures

The final selected model is highlighted in orange in every figure.

In [ ]:
SHORT_LABELS = ["1. SmallCNN\nfrom scratch", "2. ResNet-18\nfrozen", "3. ResNet-18\nfine-tuned", "3. Tuned\n(FINAL)"]
COLORS = ["#8C8C8C", "#4C72B0", "#55A868", "#DD8452"]


def bar_chart(column, title, filename):
    values = results_df[column].to_numpy()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    bars = ax.bar(SHORT_LABELS, values, color=COLORS, edgecolor="black", linewidth=0.6)
    bars[-1].set_hatch("//")
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.2%}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.08)
    ax.set_ylabel(column)
    ax.set_title(title, fontsize=13)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(COMPARISON_DIR / filename, dpi=150)
    plt.show()
    plt.close(fig)


bar_chart("Test Accuracy", "Test accuracy by approach (same 463-image test set)", "comparison_test_accuracy.png")
bar_chart("Macro F1", "Macro F1 by approach (every class weighted equally)", "comparison_macro_f1.png")

In [ ]:
# All test metrics side by side
metrics = ["Test Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
x = np.arange(len(metrics))
width = 0.2
fig, ax = plt.subplots(figsize=(12, 5.5))
for i, (label, color) in enumerate(zip(SHORT_LABELS, COLORS)):
    ax.bar(x + (i - 1.5) * width, results_df.loc[i, metrics].to_numpy(), width,
           label=label.replace("\n", " "), color=color, edgecolor="black", linewidth=0.5)
ax.set_xticks(x, metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("All test metrics by approach")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / "comparison_all_metrics.png", dpi=150)
plt.show()
plt.close(fig)

# Validation accuracy during training (one line per approach)
fig, ax = plt.subplots(figsize=(10, 5.5))
curve_sources = [
    ("1. SmallCNN from scratch", paths_used["small_cnn"].parent / "history.json", COLORS[0]),
    ("2. ResNet-18 frozen", paths_used["resnet18_frozen"].parent / "history.json", COLORS[1]),
    ("3. ResNet-18 fine-tuned", paths_used["resnet18_finetuned"].parent / "history.json", COLORS[2]),
    (f"3. Tuned, config {best_cfg['config']} (FINAL)",
     paths_used["hyperparameter_tuning"].parents[1] / "configs" / f"config_{best_cfg['config']}" / "history.json",
     COLORS[3]),
]
for label, history_path, color in curve_sources:
    if history_path.is_file():
        h = load_json(history_path)
        ax.plot(h["epoch"], h["val_acc"], marker="o", markersize=3, label=label, color=color)
    else:
        print(f"(history not found, curve skipped: {history_path})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0, 1.02)
ax.set_title("Validation accuracy during training")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(COMPARISON_DIR / "comparison_val_accuracy_curves.png", dpi=150)
plt.show()
plt.close(fig)
print(f"Saved figures to {COMPARISON_DIR}")

## 5. Effect of hyperparameter tuning

Tuning compared four learning-rate × weight-decay settings for Approach 3 and selected one **on validation accuracy only**. The tuned model's test set was used once, after selection.

In [ ]:
baseline_test = results["resnet18_finetuned"]["test"]
tuned_test = tuned["final_test_metrics"]

improvement_rows = []
for key, label in [("accuracy", "Test accuracy"), ("precision_macro", "Macro precision"),
                   ("recall_macro", "Macro recall"), ("f1_macro", "Macro F1"), ("f1_weighted", "Weighted F1")]:
    improvement_rows.append({
        "Metric": label,
        "ResNet-18 fine-tuned (baseline)": baseline_test[key],
        "Tuned ResNet-18 (final)": tuned_test[key],
        "Change (pp)": 100 * (tuned_test[key] - baseline_test[key]),
    })
improvement_df = pd.DataFrame(improvement_rows)
improvement_df.to_csv(COMPARISON_DIR / "tuning_improvement.csv", index=False)

shown = improvement_df.copy()
for col in ("ResNet-18 fine-tuned (baseline)", "Tuned ResNet-18 (final)"):
    shown[col] = shown[col].map("{:.2%}".format)
shown["Change (pp)"] = shown["Change (pp)"].map("{:+.2f}".format)
print(shown.to_string(index=False))

delta_pp = 100 * (tuned_test["accuracy"] - baseline_test["accuracy"])
print(f"\nResNet-18 Fine-Tuned baseline: {baseline_test['accuracy']:.2%}")
print(f"Tuned ResNet-18:               {tuned_test['accuracy']:.2%}")
print(f"Improvement:                   {delta_pp:+.2f} percentage points")
print(f"Selected configuration:        config {best_cfg['config']} "
      f"(lr = {best_cfg['learning_rate']:g}, weight decay = {best_cfg['weight_decay']:g}, "
      f"best validation accuracy {best_cfg['best_val_acc']:.2%})")
n_test = tuned_test["num_test_samples"]
print(f"\nOn {n_test} test images, {delta_pp:+.2f} pp is about "
      f"{round(delta_pp / 100 * n_test):+d} correctly classified images.")

## 6. Conclusion

**Final selected model:** ResNet-18 pretrained on ImageNet, fully fine-tuned, with the tuned hyperparameters (see Section 5).

**Why fine-tuning ResNet-18 performed best:**
- **Transfer learning:** ImageNet pretraining gives the network general visual features (edges, shapes, colours, textures) that a model trained from scratch on 2,153 images cannot learn well. That is the large gap between Approach 1 and Approaches 2–3.
- **Adapting the features:** the frozen backbone (Approach 2) can only re-weight fixed ImageNet features with one linear layer. Fine-tuning (Approach 3) also adapts every convolutional layer and the BatchNorm statistics to Cambodian traffic signs: their colours, Khmer text, and fine differences between similar signs. It gains most on macro F1, i.e. on the rare and visually similar classes.
- **Tuning:** a stronger weight decay (more regularisation) gave the best validation result and a further small improvement on the untouched test set.

**Trade-off:** the final model trains all 11.2M parameters, compared with 13,338 for the frozen model and 133,178 for SmallCNN, and needs more computation per epoch. On this dataset, that cost buys clearly better accuracy, especially on the less frequent classes.

**Caveat:** the test set has 463 images, so one image is about 0.22 percentage points. Small differences between the strongest models should be read with that in mind.

In [ ]:
final = results_df.iloc[-1]
summary = {
    "final_model": TUNED_LABEL,
    "selected_config": {k: best_cfg[k] for k in ("config", "learning_rate", "weight_decay", "best_val_acc")},
    "final_test_metrics": {k: tuned_test[k] for k in
                           ("accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted")},
    "baseline_test_accuracy": baseline_test["accuracy"],
    "test_accuracy_improvement_pp": delta_pp,
    "results_table": rows,
    "sources": {name: path.relative_to(RESULTS_ROOT).as_posix() for name, path in paths_used.items()},
}
with open(COMPARISON_DIR / "final_comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Final model: {TUNED_LABEL}")
print(f"Test accuracy {final['Test Accuracy']:.2%} | macro F1 {final['Macro F1']:.2%} | "
      f"weighted F1 {final['Weighted F1']:.2%} | {int(final['Trainable Parameters']):,} trainable parameters")
print(f"\nAll comparison outputs in {COMPARISON_DIR}:")
for p in sorted(COMPARISON_DIR.iterdir()):
    print(f"  {p.name}")